# Jersey colour classification in football video


In [1]:
import torch
import pandas as pd
import numpy as np

from dataset import load_manifest, get_loader
from models import load_model
from extract_embeddings import get_embeddings, extract_all_models
from classification_clustering import compare_models
from visualization import interactive_embedding_view, plot_confusion_matrix

C:\Users\KM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torchreid\reid\metrics\rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
from pathlib import Path

manifest_path = Path("dataset_v1/manifest_with_splits.csv")

if manifest_path.exists():
    df = load_manifest(manifest_path)
else:
    print("dataset_v1/manifest_with_splits.csv not found, fallback to ../dataset/manifests/*.csv")
    manifests_dir = Path("../dataset/manifests")
    parts = []
    for split_name in ["train", "val", "test"]:
        p = manifests_dir / f"{split_name}.csv"
        if not p.exists():
            continue
        part = pd.read_csv(p)
        if "split" not in part.columns:
            part["split"] = split_name
        parts.append(part)

    if not parts:
        raise FileNotFoundError(
            "No manifests found. Expected dataset_v1/manifest_with_splits.csv or ../dataset/manifests/{train,val,test}.csv"
        )

    df = pd.concat(parts, ignore_index=True)

    # Convert detection-level manifest to the format used by this notebook.
    df["crop_path"] = df["image_path"]
    df["game"] = df["source_folder"]
    df["label"] = np.where(
        df["role_name"].astype(str).str.strip().str.lower() == "goalkeeper",
        "goalkeeper",
        np.where(df["left2right"].astype(int) == 1, "team_left", "team_right"),
    )

print("dataset size:", len(df))
df.head()

dataset size: 64291


,crop_path,label,game,src_image,x1,y1,x2,y2,frame_idx,player_id,role_name,left2right,split
0,dataset_v1/crops/game_10_H1__frame_001877__i0_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,1399,522,1475,667,1877,101,Left Central Back,1,train
1,dataset_v1/crops/game_10_H1__frame_001877__i1_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,1524,290,1591,408,1877,103,Right Central Back,1,train
2,dataset_v1/crops/game_10_H1__frame_001877__i2_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,1091,874,1171,1067,1877,104,Left Back,1,train
3,dataset_v1/crops/game_10_H1__frame_001877__i3_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,77,549,126,704,1877,105,Left Midfielder,1,train
4,dataset_v1/crops/game_10_H1__frame_001877__i4_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,185,169,223,269,1877,106,Right Winger,1,train


In [4]:
df["game"].unique()[:20]

<StringArray>
['game_10_H1', 'game_10_H2', 'game_11_H1', 'game_11_H2', 'game_12_H1',
 'game_12_H2', 'game_13_H1', 'game_13_H2', 'game_14_H1', 'game_14_H2',
 'game_15_H1', 'game_15_H2', 'game_16_H1', 'game_16_H2', 'game_17_H1',
 'game_17_H2', 'game_19_H1', 'game_19_H2',  'game_1_H1',  'game_1_H2']
Length: 20, dtype: str

## game_12


In [5]:
base_game = "game_12"
df_h1 = df[df["game"] == f"{base_game}_H1"].copy()
df_h2 = df[df["game"] == f"{base_game}_H2"].copy()

print("H1:", len(df_h1), "H2:", len(df_h2), "Total:", len(df_h1) + len(df_h2))

H1: 716 H2: 695 Total: 1411


In [6]:
def flip_lr_label(label: str) -> str:
    if label == "team_left":
        return "team_right"
    if label == "team_right":
        return "team_left"
    return label

df_h2["label"] = df_h2["label"].map(flip_lr_label)

df_match = pd.concat([df_h1, df_h2], ignore_index=True)

print(df_match["label"].value_counts())

label
team_right    680
team_left     636
goalkeeper     95
Name: count, dtype: int64


In [8]:
embeddings = extract_all_models(
    df_match=df_match,
    game_id=base_game,
    device=device,
    model_names=["dino", 'osnet']
)


  модель: dino


C:\Users\KM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name vit_base_patch16_224_dino to current vit_base_patch16_224.dino.
  model = create_fn(


loading cached embeddings
  готово: shape=(1452, 768)

  модель: osnet
Successfully loaded imagenet pretrained weights from "C:\Users\KM/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"
loading cached embeddings
  готово: shape=(1452, 512)


### Metrics

In [8]:
df_report = compare_models(embeddings)

df_classification = df_report[[
    "lr_accuracy",
    "lr_macro_f1",
    "mlp_accuracy",
    "mlp_macro_f1",
    "macro_f1_delta_mlp_minus_lr",
    "best_lr",
    "best_mlp",
]].copy()

df_clustering = df_report[["cluster_acc", "cluster_macro_f1", "sil_euclidean", "sil_cosine"]].copy()
df_clustering = df_clustering.rename(columns={
    "cluster_acc": "kmeans_acc",
    "cluster_macro_f1": "kmeans_macro_f1",
    "sil_euclidean": "kmeans_sil_euclidean",
    "sil_cosine": "kmeans_sil_cosine",
})

print("Classification comparison: Logistic Regression vs 2-layer MLP")
display(df_classification)

print("Clustering metrics (k-means)")
display(df_clustering)

evaluating dino...
evaluating osnet...
Classification comparison: Logistic Regression vs 2-layer MLP


,lr_accuracy,lr_macro_f1,mlp_accuracy,mlp_macro_f1,macro_f1_delta_mlp_minus_lr,best_lr,best_mlp
model,,,,,,,
dino,0.9717,0.9551,0.9717,0.9551,0.0000,+,+
osnet,0.9576,0.9035,0.9682,0.9526,0.0491,,


Clustering metrics (k-means)


,kmeans_acc,kmeans_macro_f1,kmeans_sil_euclidean,kmeans_sil_cosine
model,,,,
dino,0.7371,0.6198,0.0964,0.1731
osnet,0.7512,0.6209,0.0934,0.1710


### Visualization

In [9]:
viz_cols = [
    "crop_path", "game", "frame_idx", "player_id",
    "x1", "y1", "x2", "y2", "source_folder", "image_file",
]

df_viz = df_match[[c for c in viz_cols if c in df_match.columns]].reset_index(drop=True)

name = "osnet"
X, y = embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1411 точек...


C:\Users\KM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_12_H1', '692', '101',
   …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_12_H1', '692', '101',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCABHACgDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDHopSKB15rwbHCJgEVBcgMjIXYICN7K2NoJ9e1dJ4

## game_50


In [4]:
base_game = "game_40"
df_h1 = df[df["game"] == f"{base_game}_H1"].copy()
df_h2 = df[df["game"] == f"{base_game}_H2"].copy()

print("H1:", len(df_h1), "H2:", len(df_h2), "Total:", len(df_h1) + len(df_h2))

H1: 740 H2: 712 Total: 1452


In [5]:
def flip_lr_label(label: str) -> str:
    if label == "team_left":
        return "team_right"
    if label == "team_right":
        return "team_left"
    return label

df_h2["label"] = df_h2["label"].map(flip_lr_label)

df_match = pd.concat([df_h1, df_h2], ignore_index=True)

print(df_match["label"].value_counts())

label
team_left     705
team_right    694
goalkeeper     53
Name: count, dtype: int64


In [6]:
embeddings = extract_all_models(
    df_match=df_match,
    game_id=base_game,
    device=device,
    model_names=["osnet", "dino", "dinov2_large", "fastreid"],
)


  модель: osnet
Successfully loaded imagenet pretrained weights from "C:\Users\KM/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"
loading cached embeddings
  готово: shape=(1452, 512)

  модель: dino


C:\Users\KM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name vit_base_patch16_224_dino to current vit_base_patch16_224.dino.
  model = create_fn(


loading cached embeddings
  готово: shape=(1452, 768)

  модель: dinov2_large


Using cache found in C:\Users\KM/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\KM/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\KM/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\KM/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


loading cached embeddings
  готово: shape=(1452, 1024)

  модель: fastreid
loading cached embeddings
  готово: shape=(1452, 2048)


### Metrics

In [8]:
df_report = compare_models(embeddings)

df_classification = df_report[[
    "lr_accuracy",
    "lr_macro_f1",
    "mlp_accuracy",
    "mlp_macro_f1",
    "macro_f1_delta_mlp_minus_lr",
    "best_lr",
    "best_mlp",
]].copy()

df_clustering = df_report[["cluster_acc", "cluster_macro_f1", "sil_euclidean", "sil_cosine"]].copy()
df_clustering = df_clustering.rename(columns={
    "cluster_acc": "kmeans_acc",
    "cluster_macro_f1": "kmeans_macro_f1",
    "sil_euclidean": "kmeans_sil_euclidean",
    "sil_cosine": "kmeans_sil_cosine",
})

print("Classification comparison: Logistic Regression vs 2-layer MLP")
display(df_classification)

print("Clustering metrics (k-means)")
display(df_clustering)

evaluating osnet...
evaluating dino...
evaluating dinov2_large...
evaluating fastreid...
Classification comparison: Logistic Regression vs 2-layer MLP


,lr_accuracy,lr_macro_f1,mlp_accuracy,mlp_macro_f1,macro_f1_delta_mlp_minus_lr,best_lr,best_mlp
model,,,,,,,
dino,0.9725,0.9320,0.9759,0.9524,0.0205,,+
fastreid,0.9622,0.8819,0.9588,0.8795,-0.0024,,
dinov2_large,0.9656,0.8292,0.9691,0.8315,0.0024,,
osnet,0.9691,0.9477,0.9553,0.8221,-0.1256,+,


Clustering metrics (k-means)


,kmeans_acc,kmeans_macro_f1,kmeans_sil_euclidean,kmeans_sil_cosine
model,,,,
dino,0.7431,0.6061,0.1023,0.1866
fastreid,0.6612,0.5268,0.0775,0.1413
dinov2_large,0.8719,0.6582,0.0667,0.1215
osnet,0.7479,0.5677,0.0886,0.1638


### Visualization

In [14]:
viz_cols = [
    "crop_path", "game", "frame_idx", "player_id",
    "x1", "y1", "x2", "y2", "source_folder", "image_file",
]

df_viz = df_match[[c for c in viz_cols if c in df_match.columns]].reset_index(drop=True)

name = "osnet"
X, y = embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1452 точек...


C:\Users\KM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
   …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCAAiABYDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDY1uzkvbDbBrA0uZXDK7FfnAByvzcf/qpYZImtYzH

## Metric Learning - Improve Embeddings

In [11]:
from metric_learning import MetricLearningTrainer, EmbeddingHead
from sklearn.model_selection import train_test_split
results_list = []
losses_to_test = ["triplet", "contrastive", "arcface"]

for model_key, (X_orig, y_orig) in embeddings.items():
    
    orig_rep = compare_models({model_key: (X_orig, y_orig)}).loc[model_key]

    for l_type in losses_to_test:
        print(f"--- Training {l_type.upper()} head for {model_key} ---")
        
        head = EmbeddingHead(in_dim=X_orig.shape[1], out_dim=128, hidden_dim=256).to(device)
        trainer = MetricLearningTrainer(model=head, device=device, learning_rate=1e-3, loss_type=l_type)
        
        trainer.train(embeddings=X_orig, labels=y_orig, epochs=15, batch_size=128, val_split=0.9)
        
        X_ref = trainer.transform(X_orig)
        ref_rep = compare_models({f"tmp": (X_ref, y_orig)}).loc["tmp"]
        
        results_list.append({
            "Base Model": model_key,
            "Loss Type": l_type,
            "orig_kmeans_acc": orig_rep['cluster_acc'],
            "refined_kmeans_acc": ref_rep['cluster_acc'],
            "orig_kmeans_f1": orig_rep['cluster_macro_f1'],
            "refined_kmeans_f1": ref_rep['cluster_macro_f1'],
            "orig_sil_eucl": orig_rep['sil_euclidean'],
            "refined_sil_eucl": ref_rep['sil_euclidean'],
            "orig_sil_cos": orig_rep['sil_cosine'],
            "refined_sil_cos": ref_rep['sil_cosine'],
        })

full_comparison_df = pd.DataFrame(results_list)

display(full_comparison_df)

evaluating osnet...
--- Training TRIPLET head for osnet ---
Training triplet loss for 15 epochs
  Train: 146 samples
  Val: 1306 samples
Epoch 1/15 - Loss: 2.7049, Val Loss: 2.7181
Epoch 2/15 - Loss: 1.6797, Val Loss: 3.1718
Epoch 3/15 - Loss: 0.7264, Val Loss: 3.7518
Epoch 4/15 - Loss: 0.1192, Val Loss: 4.3632
Epoch 5/15 - Loss: 0.0041, Val Loss: 5.0353
Epoch 6/15 - Loss: 0.0000, Val Loss: 5.7072
Epoch 7/15 - Loss: 0.0000, Val Loss: 6.3539
Epoch 8/15 - Loss: 0.0000, Val Loss: 6.9178
Epoch 9/15 - Loss: 0.0000, Val Loss: 7.4183
Epoch 10/15 - Loss: 0.0000, Val Loss: 7.8537
Epoch 11/15 - Loss: 0.0000, Val Loss: 8.2478
Epoch 12/15 - Loss: 0.0000, Val Loss: 8.5473
Epoch 13/15 - Loss: 0.0000, Val Loss: 8.8229
Epoch 14/15 - Loss: 0.0000, Val Loss: 9.0472
Epoch 15/15 - Loss: 0.0000, Val Loss: 9.2814
evaluating tmp...
--- Training CONTRASTIVE head for osnet ---
Training contrastive loss for 15 epochs
  Train: 146 samples
  Val: 1306 samples
Epoch 1/15 - Loss: 0.2811, Val Loss: 0.1019
Epoch 2/15

,Base Model,Loss Type,orig_kmeans_acc,refined_kmeans_acc,orig_kmeans_f1,refined_kmeans_f1,orig_sil_eucl,refined_sil_eucl,orig_sil_cos,refined_sil_cos
0,osnet,triplet,0.7479,0.9690,0.5677,0.9133,0.0886,0.5348,0.1638,0.7605
1,osnet,contrastive,0.7479,0.9745,0.5677,0.9464,0.0886,0.7368,0.1638,0.9101
2,osnet,arcface,0.7479,0.9318,0.5677,0.8282,0.0886,0.3491,0.1638,0.5569
3,dino,triplet,0.7431,0.9697,0.6061,0.9156,0.1023,0.5937,0.1866,0.8125
4,dino,contrastive,0.7431,0.9821,0.6061,0.9723,0.1023,0.8116,0.1866,0.9437
5,dino,arcface,0.7431,0.9766,0.6061,0.9574,0.1023,0.5178,0.1866,0.7460
6,dinov2_large,triplet,0.8719,0.9690,0.6582,0.9122,0.0667,0.4946,0.1215,0.7238
7,dinov2_large,contrastive,0.8719,0.9545,0.6582,0.8638,0.0667,0.6835,0.1215,0.8615
8,dinov2_large,arcface,0.8719,0.8430,0.6582,0.7117,0.0667,0.3673,0.1215,0.5584
9,fastreid,triplet,0.6612,0.8347,0.5268,0.6421,0.0775,0.4530,0.1413,0.6171


### Visualization: Embedding Space Comparison

In [17]:
model_name = "osnet"
X_orig, y_orig = embeddings[model_name]

refined_embeddings = {
    "osnet_original": (X_orig, y_orig)
}

for l_type in losses_to_test:
    print(f"\n--- Training {l_type.upper()} for {model_name} ---")
    

    head = EmbeddingHead(in_dim=X_orig.shape[1], out_dim=128, hidden_dim=256).to(device)
    

    trainer = MetricLearningTrainer(model=head, device=device, learning_rate=1e-3, loss_type=l_type)
    
    trainer.train(embeddings=X_orig, labels=y_orig, epochs=15, batch_size=128, val_split=0.8)
    

    X_refined = trainer.transform(X_orig)
    refined_embeddings[f"{model_name}_{l_type}"] = (X_refined, y_orig)


--- Training TRIPLET for osnet ---
Training triplet loss for 15 epochs
  Train: 291 samples
  Val: 1161 samples
Epoch 1/15 - Loss: 2.7718, Val Loss: 2.4396
Epoch 2/15 - Loss: 1.8799, Val Loss: 2.9682
Epoch 3/15 - Loss: 1.2567, Val Loss: 3.6163
Epoch 4/15 - Loss: 0.5646, Val Loss: 4.3717
Epoch 5/15 - Loss: 0.1058, Val Loss: 4.9664
Epoch 6/15 - Loss: 0.0302, Val Loss: 5.5023
Epoch 7/15 - Loss: 0.0116, Val Loss: 6.1678
Epoch 8/15 - Loss: 0.0042, Val Loss: 6.8023
Epoch 9/15 - Loss: 0.0000, Val Loss: 7.3027
Epoch 10/15 - Loss: 0.0003, Val Loss: 7.8051
Epoch 11/15 - Loss: 0.0000, Val Loss: 8.1405
Epoch 12/15 - Loss: 0.0003, Val Loss: 8.4062
Epoch 13/15 - Loss: 0.0015, Val Loss: 8.6007
Epoch 14/15 - Loss: 0.0000, Val Loss: 8.7655
Epoch 15/15 - Loss: 0.0000, Val Loss: 8.8246

--- Training CONTRASTIVE for osnet ---
Training contrastive loss for 15 epochs
  Train: 291 samples
  Val: 1161 samples
Epoch 1/15 - Loss: 0.2307, Val Loss: 0.0777
Epoch 2/15 - Loss: 0.0721, Val Loss: 0.0529
Epoch 3/15 -

In [19]:
viz_cols = [
    "crop_path", "game", "frame_idx", "player_id",
    "x1", "y1", "x2", "y2", "source_folder", "image_file",
]

df_viz = df_match[[c for c in viz_cols if c in df_match.columns]].reset_index(drop=True)

name = "osnet_original"
X, y = refined_embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1452 точек...


C:\Users\KM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
   …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCAAiABYDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDY1uzkvbDbBrA0uZXDK7FfnAByvzcf/qpYZImtYzH

In [20]:
viz_cols = [
    "crop_path", "game", "frame_idx", "player_id",
    "x1", "y1", "x2", "y2", "source_folder", "image_file",
]

df_viz = df_match[[c for c in viz_cols if c in df_match.columns]].reset_index(drop=True)

name = "osnet_triplet"
X, y = refined_embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1452 точек...


C:\Users\KM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
   …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCAAiABYDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDY1uzkvbDbBrA0uZXDK7FfnAByvzcf/qpYZImtYzH

In [21]:
viz_cols = [
    "crop_path", "game", "frame_idx", "player_id",
    "x1", "y1", "x2", "y2", "source_folder", "image_file",
]

df_viz = df_match[[c for c in viz_cols if c in df_match.columns]].reset_index(drop=True)

name = "osnet_contrastive"
X, y = refined_embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1452 точек...


C:\Users\KM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
   …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCAAiABYDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDY1uzkvbDbBrA0uZXDK7FfnAByvzcf/qpYZImtYzH

In [22]:
viz_cols = [
    "crop_path", "game", "frame_idx", "player_id",
    "x1", "y1", "x2", "y2", "source_folder", "image_file",
]

df_viz = df_match[[c for c in viz_cols if c in df_match.columns]].reset_index(drop=True)

name = "osnet_arcface"
X, y = refined_embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1452 точек...


C:\Users\KM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
   …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCAAiABYDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDY1uzkvbDbBrA0uZXDK7FfnAByvzcf/qpYZImtYzH

## Unsupervised Metric Learning - SimCLR

In [7]:
from metric_learning import SimCLRTrainer, EmbeddingHead
results_list = []

for model_key, (X_orig, y_orig) in embeddings.items():
    
    orig_rep = compare_models({model_key: (X_orig, y_orig)}).loc[model_key]

    print(f"--- Training SimCLR head for {model_key} ---")
    
    head = EmbeddingHead(in_dim=X_orig.shape[1], out_dim=128, hidden_dim=256).to(device)
    trainer = SimCLRTrainer(model=head, device=device, learning_rate=1e-3, temperature=1.0)
    
    trainer.train(embeddings=X_orig, epochs=15, batch_size=128, val_split=0.1)
    
    X_ref = trainer.transform(X_orig)
    ref_rep = compare_models({f"tmp": (X_ref, y_orig)}).loc["tmp"]
    
    results_list.append({
        "Base Model": model_key,
        "Loss Type": "simclr",
        "orig_kmeans_acc": orig_rep['cluster_acc'],
        "refined_kmeans_acc": ref_rep['cluster_acc'],
        "orig_kmeans_f1": orig_rep['cluster_macro_f1'],
        "refined_kmeans_f1": ref_rep['cluster_macro_f1'],
        "orig_sil_eucl": orig_rep['sil_euclidean'],
        "refined_sil_eucl": ref_rep['sil_euclidean'],
        "orig_sil_cos": orig_rep['sil_cosine'],
        "refined_sil_cos": ref_rep['sil_cosine'],
    })

simclr_comparison_df = pd.DataFrame(results_list)

display(simclr_comparison_df)

evaluating osnet...
--- Training SimCLR head for osnet ---
Training SimCLR: 1307 train, 145 val
Epoch 1/15 - Loss: 4.7404, Val Loss: 4.9665
Epoch 2/15 - Loss: 4.6036, Val Loss: 4.8495
Epoch 3/15 - Loss: 4.5720, Val Loss: 4.7948
Epoch 4/15 - Loss: 4.5536, Val Loss: 4.7772
Epoch 5/15 - Loss: 4.5423, Val Loss: 4.7704
Epoch 6/15 - Loss: 4.5350, Val Loss: 4.7663
Epoch 7/15 - Loss: 4.5309, Val Loss: 4.7631
Epoch 8/15 - Loss: 4.5264, Val Loss: 4.7574
Epoch 9/15 - Loss: 4.5236, Val Loss: 4.7571
Epoch 10/15 - Loss: 4.5206, Val Loss: 4.7553
Epoch 11/15 - Loss: 4.5184, Val Loss: 4.7532
Epoch 12/15 - Loss: 4.5173, Val Loss: 4.7507
Epoch 13/15 - Loss: 4.5153, Val Loss: 4.7493
Epoch 14/15 - Loss: 4.5145, Val Loss: 4.7455
Epoch 15/15 - Loss: 4.5128, Val Loss: 4.7481
evaluating tmp...
evaluating dino...
--- Training SimCLR head for dino ---
Training SimCLR: 1307 train, 145 val
Epoch 1/15 - Loss: 4.7035, Val Loss: 4.8357
Epoch 2/15 - Loss: 4.5733, Val Loss: 4.7856
Epoch 3/15 - Loss: 4.5452, Val Loss: 4

,Base Model,Loss Type,orig_kmeans_acc,refined_kmeans_acc,orig_kmeans_f1,refined_kmeans_f1,orig_sil_eucl,refined_sil_eucl,orig_sil_cos,refined_sil_cos
0,osnet,simclr,0.7479,0.7314,0.5677,0.6057,0.0886,0.0866,0.1638,0.1447
1,dino,simclr,0.7431,0.6191,0.6061,0.4755,0.1023,0.0658,0.1866,0.1129
2,dinov2_large,simclr,0.8719,0.6846,0.6582,0.5807,0.0667,0.0687,0.1215,0.1188
3,fastreid,simclr,0.6612,0.4463,0.5268,0.3712,0.0775,0.0542,0.1413,0.0941


### Visualization: SimCLR Embedding Space Comparison

In [8]:
model_name = "osnet"
X_orig, y_orig = embeddings[model_name]

simclr_embeddings = {
    "osnet_original": (X_orig, y_orig)
}

print(f"\n--- Training SimCLR for {model_name} ---")

head = EmbeddingHead(in_dim=X_orig.shape[1], out_dim=128, hidden_dim=256).to(device)

trainer = SimCLRTrainer(model=head, device=device, learning_rate=1e-3, temperature=1.0)

trainer.train(embeddings=X_orig, epochs=15, batch_size=128, val_split=0.1)

X_simclr = trainer.transform(X_orig)
simclr_embeddings[f"{model_name}_simclr"] = (X_simclr, y_orig)


--- Training SimCLR for osnet ---
Training SimCLR: 1307 train, 145 val
Epoch 1/15 - Loss: 4.7346, Val Loss: 4.9309
Epoch 2/15 - Loss: 4.6016, Val Loss: 4.8285
Epoch 3/15 - Loss: 4.5700, Val Loss: 4.7982
Epoch 4/15 - Loss: 4.5546, Val Loss: 4.7814
Epoch 5/15 - Loss: 4.5434, Val Loss: 4.7762
Epoch 6/15 - Loss: 4.5375, Val Loss: 4.7676
Epoch 7/15 - Loss: 4.5319, Val Loss: 4.7627
Epoch 8/15 - Loss: 4.5273, Val Loss: 4.7606
Epoch 9/15 - Loss: 4.5242, Val Loss: 4.7592
Epoch 10/15 - Loss: 4.5216, Val Loss: 4.7579
Epoch 11/15 - Loss: 4.5190, Val Loss: 4.7542
Epoch 12/15 - Loss: 4.5172, Val Loss: 4.7523
Epoch 13/15 - Loss: 4.5157, Val Loss: 4.7488
Epoch 14/15 - Loss: 4.5139, Val Loss: 4.7481
Epoch 15/15 - Loss: 4.5122, Val Loss: 4.7462


In [9]:
viz_cols = [
    "crop_path", "game", "frame_idx", "player_id",
    "x1", "y1", "x2", "y2", "source_folder", "image_file",
]

df_viz = df_match[[c for c in viz_cols if c in df_match.columns]].reset_index(drop=True)

name = "osnet_original"
X, y = simclr_embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1452 точек...


C:\Users\KM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
   …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCAAiABYDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDY1uzkvbDbBrA0uZXDK7FfnAByvzcf/qpYZImtYzH

In [10]:
viz_cols = [
    "crop_path", "game", "frame_idx", "player_id",
    "x1", "y1", "x2", "y2", "source_folder", "image_file",
]

df_viz = df_match[[c for c in viz_cols if c in df_match.columns]].reset_index(drop=True)

name = "osnet_simclr"
X, y = simclr_embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1452 точек...


C:\Users\KM\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
   …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCAAiABYDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDY1uzkvbDbBrA0uZXDK7FfnAByvzcf/qpYZImtYzH